# AncestryClassifier — Colab Training

Run preprocessing locally with Snakemake first:
```bash
snakemake --cores 4 prepare_training_data simulate_admixed
```
Upload `data/dataset.h5` and `data/admixed_test.h5` to Google Drive, then set `DRIVE_DIR` below.

**After any runtime restart: re-run Cell 1 (config) before running any other cell.**

In [1]:
# ── Cell 1: config — re-run this first after every runtime restart ─────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR     = '/content/drive/MyDrive/gene461'          # <-- change if needed
REPO          = '/content/gene_461_final_project'
WINDOW_SIZE   = 1000

DATA_H5       = f'{DRIVE_DIR}/dataset.h5'
ADMIXED_H5    = f'{DRIVE_DIR}/admixed_test.h5'
CKPT_OUT      = f'{DRIVE_DIR}/checkpoints/best_model.pt'
CONFUSION_OUT = f'{DRIVE_DIR}/confusion_matrix.png'
KARYOGRAM_OUT = f'{DRIVE_DIR}/lai_karyogram.png'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ── Cell 2: one-time setup (clone repo + install deps) ────────────────────
import subprocess, os

if not os.path.exists(REPO):
    subprocess.run(
        ['git', 'clone', 'https://github.com/aszatrowski/gene_461_final_project', REPO],
        check=True
    )
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

%pip install -q torch h5py numpy pandas scikit-learn matplotlib wandb

In [3]:
# ── Cell 3: verify GPU ─────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CUDA available: True
GPU: Tesla T4


In [ ]:
# ── Cell 4: baseline train (needs Cell 1) ─────────────────────────────────
import os
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

!python {REPO}/scripts/train.py \
    --data        {DATA_H5}    \
    --output      {CKPT_OUT}   \
    --window-size {WINDOW_SIZE} \
    --epochs      25           \
    --batch-size  512          \
    --num-workers 2

Device: cuda
Loading training data ...


^C


In [4]:
# ── Cell 5: W&B login (one-time per session) ───────────────────────────────
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: aszatrowski (aszatrowski-university-of-chicago) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
# ── Cell 6: sweep config ───────────────────────────────────────────────────
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'window_size': {'values': [2500, 5000, 10000, 20000]},
        'lr':          {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1e-2},
        'conv_arch':   {'values': [
            # undilated — ERF ~23–55 SNPs
            '32,64_7,5',
            '64,128_7,5',
            '32,64,128_7,5,3',
            # dilated — exponential dilation expands ERF to cover the full window
            '64,128_7,7_1,4',
            '32,64,128_7,7,7_1,4,16',  # ERF ~1600 SNPs
            '32,64,128_7,7,7_4,16,64',  # ERF ~1600 SNPs
            '32,64,128_14,14,14_1,4,16',  
            '32,64,128_7,7,7_1,8,32',  
            '64,128,256_7,7,7_1,4,16',
            '64,128,256_7,7,7_1,8,32',
        ]},
        'global_pool': {'values': [False]},
        'dropout':     {'distribution': 'uniform', 'min': 0.1, 'max': 0.5},
    },
}

sweep_id = wandb.sweep(sweep_config, project='ancestry_cnn')
print('Sweep ID:', sweep_id)

Create sweep with ID: 9s7owsim
Sweep URL: https://wandb.ai/aszatrowski-university-of-chicago/ancestry_cnn/sweeps/9s7owsim
Sweep ID: 9s7owsim


In [ ]:
# ── Cell 7: run sweep agent (needs Cells 1, 3, 5, 6) ──────────────────────
import sys, importlib.util

def _load_module(name, path):
    """Load a .py file from an explicit path, registering it in sys.modules
    so that internal cross-imports (e.g. train.py → model.py) resolve correctly."""
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

_load_module('model', f'{REPO}/scripts/model.py')
_train = _load_module('train', f'{REPO}/scripts/train.py')
run_training    = _train.run_training
parse_conv_arch = _train.parse_conv_arch

SWEEP_EPOCHS = 10
SWEEP_COUNT  = 20

def sweep_fn():
    with wandb.init() as run:
        w = dict(run.config)
        channels, kernels, dilations = parse_conv_arch(w['conv_arch'])
        cfg = {
            'window_size':    w['window_size'],
            'lr':             w['lr'],
            'conv_channels':  channels,
            'kernel_sizes':   kernels,
            'dilation_rates': dilations,
            'global_pool':    w['global_pool'],
            'dropout':        w['dropout'],
            'epochs':         SWEEP_EPOCHS,
            'batch_size':     512,
            'num_workers':    2,
            'use_wandb':      True,
        }
        run_training(cfg, DATA_H5, None, device)

wandb.agent(sweep_id, sweep_fn, count=SWEEP_COUNT)

wandb: Agent Starting Run: qt0xuk7x with config:
wandb: 	conv_arch: 64,128_7,5
wandb: 	dropout: 0.17794453568392066
wandb: 	global_pool: False
wandb: 	lr: 0.00031778452070038173
wandb: 	window_size: 20000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  train windows: 14,016  |  val windows: 3,008
  params: 41,003,525
Epoch   1/10  loss=9.4144  train_acc=0.2836  val_acc=0.2979
Epoch   2/10  loss=1.2903  train_acc=0.4737  val_acc=0.6210
Epoch   3/10  loss=1.1601  train_acc=0.5027  val_acc=0.6250
Epoch   4/10  loss=1.1148  train_acc=0.5136  val_acc=0.6473
Epoch   5/10  loss=1.0746  train_acc=0.5229  val_acc=0.6642
Epoch   6/10  loss=1.0471  train_acc=0.5338  val_acc=0.6735
Epoch   7/10  loss=1.0370  train_acc=0.5398  val_acc=0.6732
Epoch   8/10  loss=1.0254  train_acc=0.5459  val_acc=0.6822
Epoch   9/10  loss=1.0050  train_acc=0.5628  val_acc=0.6812
Epoch  10/10  loss=1.0090  train_acc=0.5591  val_acc=0.6855

Best val accuracy: 0.6855


epoch,▁▂▃▃▄▅▆▆▇█
loss,█▁▁▁▁▁▁▁▁▁
train_acc,▁▆▆▇▇▇▇███
val_acc,▁▇▇▇██████
best_val_acc,0.68551
epoch,10
loss,1.00901
train_acc,0.55915
val_acc,0.68551


wandb: Agent Starting Run: ghb8y0cy with config:
wandb: 	conv_arch: 32,64,128_7,5,3
wandb: 	dropout: 0.1753702980501727
wandb: 	global_pool: False
wandb: 	lr: 0.001993573575743798
wandb: 	window_size: 2500
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


In [ ]:
# ── Cell 8: retrain best config for full 25 epochs ────────────────────────
api   = wandb.Api()
sweep = api.sweep(f'aszatrowski-university-of-chicago/ancestry_cnn/{sweep_id}')
best  = max(sweep.runs, key=lambda r: r.summary.get('val_acc', 0))
print('Best run:', best.name)
print('Config:  ', dict(best.config))
print(f'val_acc:  {best.summary["val_acc"]:.4f}')

bc = dict(best.config)
channels, kernels, dilations = parse_conv_arch(bc['conv_arch'])
best_cfg = {
    'window_size':    bc['window_size'],
    'lr':             bc['lr'],
    'conv_channels':  channels,
    'kernel_sizes':   kernels,
    'dilation_rates': dilations,
    'global_pool':    bc['global_pool'],
    'dropout':        bc['dropout'],
    'epochs':         25,
    'batch_size':     512,
    'num_workers':    2,
    'use_wandb':      True,
}
with wandb.init(project='ancestry_cnn', config=best_cfg, name='best_retrain'):
    run_training(best_cfg, DATA_H5, CKPT_OUT, device)

In [ ]:
# ── Cell 9: evaluate (needs Cell 1 only — safe after a restart) ───────────
!python {REPO}/scripts/evaluate.py \
    --data        {DATA_H5}       \
    --admixed     {ADMIXED_H5}    \
    --checkpoint  {CKPT_OUT}      \
    --confusion   {CONFUSION_OUT} \
    --karyogram   {KARYOGRAM_OUT} \
    --window-size {WINDOW_SIZE}

In [ ]:
# ── Cell 10: display results (needs Cell 1 only) ──────────────────────────
from IPython.display import Image, display
display(Image(CONFUSION_OUT))
display(Image(KARYOGRAM_OUT))